# Simple Classification
1. Loop through each spanned annotations and try to reverse find the source text that contains this annotation (positive chunk) - classify if a chunk contains a span
2. Randomly find chunks that do not have annotations (negative chunks)
3. Turn source text to vectors using Roberta
4. Train classifier
---
1. Load data, chunk, inspect
2. Build one spans table (contract_id, category_id, start, end) for the 10 categories
3. Label chunks per category
4. Check label counts per category
5. Split train/val by contract
6. Embed with RoBERTa
7. Train and evaluate


In [1]:
from utils import load_file_from_github, normalize_cuad

raw_test = load_file_from_github("data/cuad/test.json")
raw_train = load_file_from_github("data/cuad/train_separate_questions.json")
training_data = normalize_cuad(raw_train["data"], cuad=True)
test_data = normalize_cuad(raw_test["data"], cuad=True)

final_cat_names = [
    "Governing Law",
    "Renewal Term",
    "Revenue/Profit Sharing",
    "Cap On Liability",
    "Uncapped Liability",
    "Termination For Convenience",
    "Anti-Assignment",
    "Audit Rights",
    "License Grant",
    "Exclusivity"
]
final_ids = training_data.categories.loc[training_data.categories.category_name.isin(final_cat_names), 'category_id']

def get_final_10(data):

    # get all annotation sets in the final 10 categories
    sets = data.annotation_sets[data.annotation_sets.category_id.isin(final_ids)]

    # get all documents corresponding to the final annotation sets
    documents = data.documents[data.documents["contract_id"].isin(sets["contract_id"])]

    # get all spans in the final 10 categories
    spans = data.spans.merge(sets[["annotation_set_id", "category_id", "contract_id"]], on="annotation_set_id")

    return sets, documents, spans

final_sets, final_documents, final_spans = get_final_10(training_data)
test_sets, test_documents, test_spans = get_final_10(test_data)

final_spans.head()

,span_id,annotation_set_id,source_qa_id,answer_text,answer_start,answer_end,category_id,contract_id
0,contract_0001__renewal_term__span_000,contract_0001__renewal_term,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,If Distributor comp...,5543,5881,renewal_term,contract_0001
1,contract_0001__governing_law__span_000,contract_0001__governing_law,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,This Agreement is to be construed according to...,52061,52151,governing_law,contract_0001
2,contract_0001__exclusivity__span_000,contract_0001__exclusivity,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,The Company appoints the Distributor as an ex...,1854,2004,exclusivity,contract_0001
3,contract_0001__exclusivity__span_001,contract_0001__exclusivity,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Company hereby appoints ...,2112,2611,exclusivity,contract_0001
4,contract_0001__exclusivity__span_002,contract_0001__exclusivity,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,The Distributor shall not order or ...,12390,12503,exclusivity,contract_0001


# Chunk Document Context

- Chunking by paragraph
- DistilRoBERTa tokenizer
- Chunk Size = 512 tokens
- Overlap = 64 tokens
- Result: 10,841 Chunks

In [2]:
from chunking_helper import create_paragraph_chunks
from transformers import AutoTokenizer

MODEL_ID = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

chunks = create_paragraph_chunks(final_documents, tokenizer, chunk_size=512)

# add contract text to chunks
chunks["text"] = chunks.apply(
    lambda row: final_documents.loc[
        final_documents["contract_id"] == row["contract_id"], "context"
    ].iloc[0][row["char_start"]:row["char_end"]],
    axis=1
)

chunks.head()

/Users/amiebinan/Desktop/Internships/Accenture-1O-contract-review-challenge/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (803 > 512). Running this sequence through the model will result in indexing errors


,method,contract_id,context_group_id,chunk_id,chunk_number,char_start,char_end,num_tokens,text
0,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0000,0,0,2056,463,EXHIBIT 10.6\n\n ...
1,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0001,1,2076,3126,512,1.1 Grant and Acceptance. Company h...
2,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0002,2,2979,3717,359,such customers in the Market to those custom...
3,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0003,3,3737,4788,512,1.2 License. The Company hereby grants ...
4,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0004,4,4658,5228,277,sublicense the Names or to do ...


In [3]:
chunks["num_tokens"].max()

np.int64(512)

# Positives

Loop through each spanned annotations and try to reverse find the source text that contains this annotation (positive chunk) - classify if a chunk contains a span

In [4]:
import pandas as pd

def find_positives(chunks):
    chunks_by_contract_idx = chunks.groupby("contract_id").indices
    start, end = chunks.char_start.values, chunks.char_end.values

    hits, pos_found = [], 0
    for row in final_spans.itertuples():
        chunk_idxes = chunks_by_contract_idx.get(row.contract_id)

        # if contract has no annotation sets
        if chunk_idxes is None:
            continue

        # matches chunk indices where span start and end are contained in the chunk
        matches = chunk_idxes[(start[chunk_idxes] <= row.answer_start)
                              & (row.answer_end <= end[chunk_idxes])]
        pos_found += len(matches) > 0

        # saves the chunk_id and their category of each match
        hits += [(row.category_id, chunks.chunk_id.iat[i]) for i in matches]

    positives = pd.DataFrame(hits, columns=["category_id", "chunk_id"]).drop_duplicates()
    return positives, pos_found

positives, pos_found = find_positives(chunks)
print(f"{pos_found}/{len(final_spans)} spans matched a chunk")
positives.groupby("category_id").size().reset_index(name="count")

3721/3823 spans matched a chunk


,category_id,count
0,anti_assignment,398
1,audit_rights,332
2,cap_on_liability,401
3,exclusivity,244
4,governing_law,376
5,license_grant,472
6,renewal_term,156
7,revenue_profit_sharing,218
8,termination_for_convenience,177
9,uncapped_liability,113


# Negatives

Randomly find chunks that do not have annotations (negative chunks)

In [5]:
category_ids = positives["category_id"].unique().tolist()

def get_chunk_categories(chunks):

    # cartesian product of chunk_id and category_id
    chunk_categories = pd.merge(chunks["chunk_id"], pd.Series(category_ids, name="category_id"), how="cross")

    # mark all tuples in positives as chunk contains category
    chunk_categories = chunk_categories.merge(
        positives.assign(is_contained=True),
        on=["chunk_id", "category_id"], how="left",
    )
    chunk_categories["is_contained"] = chunk_categories["is_contained"].fillna(False).astype(bool)

    return chunk_categories

chunk_categories = get_chunk_categories(chunks)

# negatives are all tuples where is_contained == False
negatives = chunk_categories.loc[~chunk_categories["is_contained"], ["chunk_id", "category_id"]]
negatives

,chunk_id,category_id
0,contract_0001__chunk_0000,renewal_term
1,contract_0001__chunk_0000,governing_law
3,contract_0001__chunk_0000,anti_assignment
4,contract_0001__chunk_0000,license_grant
5,contract_0001__chunk_0000,audit_rights
...,...,...
137135,contract_0408__chunk_0013,audit_rights
137136,contract_0408__chunk_0013,uncapped_liability
137137,contract_0408__chunk_0013,cap_on_liability
137138,contract_0408__chunk_0013,termination_for_convenience


In [6]:
negatives.groupby("category_id").size().reset_index(name="count")

,category_id,count
0,anti_assignment,13316
1,audit_rights,13382
2,cap_on_liability,13313
3,exclusivity,13470
4,governing_law,13338
5,license_grant,13242
6,renewal_term,13558
7,revenue_profit_sharing,13496
8,termination_for_convenience,13537
9,uncapped_liability,13601


In [7]:
assert len(chunk_categories) == len(chunks) * len(category_ids)
assert chunk_categories["is_contained"].sum() == len(positives)
assert len(negatives) + len(positives) == len(chunk_categories)

print("Everything adds up!")

Everything adds up!


# Imbalance

In [8]:
imbalance = (
    chunk_categories.groupby("category_id")["is_contained"]
    .agg(n_pos="sum", n_total="size")
    .assign(
        n_neg=lambda d: d.n_total - d.n_pos,
        pos_rate=lambda d: d.n_pos / d.n_total,
        neg_per_pos=lambda d: (d.n_total - d.n_pos) / d.n_pos,
    )
    .sort_values("pos_rate")
)
imbalance.round(4)

,n_pos,n_total,n_neg,pos_rate,neg_per_pos
category_id,,,,,
uncapped_liability,113,13714,13601,0.0082,120.3628
renewal_term,156,13714,13558,0.0114,86.9103
termination_for_convenience,177,13714,13537,0.0129,76.4802
revenue_profit_sharing,218,13714,13496,0.0159,61.9083
exclusivity,244,13714,13470,0.0178,55.2049
audit_rights,332,13714,13382,0.0242,40.3072
governing_law,376,13714,13338,0.0274,35.4734
anti_assignment,398,13714,13316,0.0290,33.4573
cap_on_liability,401,13714,13313,0.0292,33.1995


In [9]:
imbalance["pos_rate"].sum()

np.float64(0.2105148023917165)

# Vector Embeddings
Turn source text to vectors using Roberta

In [10]:
# freeze chunks for use in Google Colab
chunks[["chunk_id", "text"]].to_parquet("train_chunks.parquet")

In [30]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

# Mean Pooling - Take attention mask into account for correct averaging
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] # First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained('sentence-transformers/all-distilroberta-v1')

chunk_text = chunks["text"].tolist()

# tokenize sentences
encoded_input = tokenizer(chunk_text, padding=True, truncation=True, return_tensors='pt')

# Compute token embeddings
with torch.no_grad():
    model_output = model(**encoded_input)

# Perform pooling
chunk_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])

# Normalize embeddings
chunk_embeddings = F.normalize(chunk_embeddings, p=2, dim=1)

print("Sentence embeddings:")
print(chunk_embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7536.87it/s]


KeyboardInterrupt: 

In [ ]:
chunks.head()

Import vector embeddings from Google Colab notebook

In [31]:
import numpy as np
embeddings = np.load("train_embeddings.npy")
embeddings.shape

(2984, 768)

In [32]:
chunks = chunks.reset_index(drop=True)
chunks["embeddings"] = list(embeddings)
chunks.head()

,method,contract_id,context_group_id,chunk_id,chunk_number,char_start,char_end,num_tokens,text,embeddings
0,paragraph_based,contract_0001,context_11ef4095ae,contract_0001__chunk_0000,0,0,2063,468,Exhibit 10.16 SUPPLY CONTRACT Contract No: Dat...,"[0.0025928824, 0.005389281, 0.002025629, 0.011..."
1,paragraph_based,contract_0001,context_11ef4095ae,contract_0001__chunk_0001,1,2069,3587,346,3. GOODS AND COUNTRY OF ORIGIN: 4. Specific or...,"[-0.046105206, -0.01579429, -0.009005282, -0.0..."
2,paragraph_based,contract_0001,context_11ef4095ae,contract_0001__chunk_0002,2,3593,5195,371,"9. PORT OF DESTINATION: SHENZHEN, GUANGDONG, C...","[-0.020681655, -0.026813367, -0.018308526, -0...."
3,paragraph_based,contract_0001,context_11ef4095ae,contract_0001__chunk_0003,3,5201,7629,512,12.2 (1) Invoice in 3 originals indicating con...,"[-0.03207245, 0.0212491, -0.0131449, -0.033636..."
4,paragraph_based,contract_0001,context_11ef4095ae,contract_0001__chunk_0004,4,7311,7909,129,by the Sellers. The Certificate so issued shal...,"[-9.912056e-05, 0.030205805, -0.030177, -0.031..."


# Logistic Regression


In [33]:
from sklearn.linear_model import LogisticRegression

row_of = pd.Series(np.arange(len(chunks)), index=chunks["chunk_id"])

def get_embeddings(chunk_ids):
    """
    Retrieves all embeddings for a list of chunk_ids
    :param chunk_ids: chunks to retrieve embeddings for
    :return: pd.Series of embeddings for chunk_ids
    """
    return embeddings[row_of[chunk_ids].values]

def fit_category(category):
    """
    Trains a Logistic Regression model on a given category
    :param category: the category label
    :return: the trained model
    """
    chunk_key = chunk_categories[(chunk_categories["category_id"] == category)]
    X = get_embeddings(chunk_key["chunk_id"])
    y = chunk_key["is_contained"].values.astype(int)
    clf = LogisticRegression()
    return clf.fit(X, y)

models = {cat: fit_category(cat) for cat in final_ids}

In [16]:
chunk_categories

,chunk_id,category_id,is_contained
0,contract_0001__chunk_0000,renewal_term,False
1,contract_0001__chunk_0000,governing_law,False
2,contract_0001__chunk_0000,exclusivity,True
3,contract_0001__chunk_0000,anti_assignment,False
4,contract_0001__chunk_0000,license_grant,False
...,...,...,...
108405,contract_0408__chunk_0013,audit_rights,False
108406,contract_0408__chunk_0013,uncapped_liability,False
108407,contract_0408__chunk_0013,cap_on_liability,False
108408,contract_0408__chunk_0013,termination_for_convenience,False


# Testing

We are using an 64/20 split of training and test data, ommitting validation in the meantime.

In [18]:
test_chunks = create_paragraph_chunks(test_documents, tokenizer, chunk_size=512)

test_chunks["text"] = test_chunks.apply(
    lambda row: test_documents.loc[
        test_documents["contract_id"] == row["contract_id"], "context"
    ].iloc[0][row["char_start"]:row["char_end"]],
    axis=1
)

test_chunk_categories = get_chunk_categories(test_chunks)

test_negatives = test_chunk_categories.loc[~test_chunk_categories["is_contained"], ["chunk_id", "category_id"]]

# freeze chunks for use in Google Colab
test_chunks[["chunk_id", "text"]].to_parquet("test_chunks.parquet")

In [20]:
test_embeddings = np.load("test_embeddings.npy")
test_chunks = test_chunks.reset_index(drop=True)
test_chunks["embeddings"] = list(test_embeddings)